In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
from transformers import BertTokenizer

model_name = 'bert-base-uncased'

tokenizer = BertTokenizer.from_pretrained(model_name)
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [3]:
text = 'Here is the sentence I want embedding for'
tokenizer.encode(text)

[101, 2182, 2003, 1996, 6251, 1045, 2215, 7861, 8270, 4667, 2005, 102]

In [4]:
tokenizer.tokenize(text)

['here', 'is', 'the', 'sentence', 'i', 'want', 'em', '##bed', '##ding', 'for']

In [5]:
tokenizer(text)

{'input_ids': [101, 2182, 2003, 1996, 6251, 1045, 2215, 7861, 8270, 4667, 2005, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [6]:
from transformers import BertForMaskedLM

model = BertForMaskedLM.from_pretrained(model_name)
model

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [7]:
text = 'Soccer is a really fun [MASK]'

tokens = tokenizer.tokenize(text)
print(tokens)

inputs = tokenizer(text, return_tensors='pt')
inputs

['soccer', 'is', 'a', 'really', 'fun', '[MASK]']


{'input_ids': tensor([[ 101, 4715, 2003, 1037, 2428, 4569,  103,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [8]:
print(tokenizer.cls_token_id)
print(tokenizer.sep_token_id)
print(tokenizer.mask_token_id)

101
102
103


In [9]:
output = model(**inputs)
output

MaskedLMOutput(loss=None, logits=tensor([[[ -6.8576,  -6.8062,  -6.7971,  ...,  -6.1423,  -6.0306,  -4.0758],
         [ -6.0743,  -6.4033,  -6.2322,  ...,  -5.2093,  -5.5435,  -2.7450],
         [-12.0165, -11.6791, -11.8348,  ..., -10.5101,  -8.9052,  -8.1035],
         ...,
         [ -8.2386,  -8.6850,  -8.7986,  ...,  -8.6419,  -6.8287,  -8.7747],
         [ -8.8995,  -8.6852,  -8.9502,  ...,  -8.9231,  -9.1263,  -3.7561],
         [-12.4644, -12.6299, -12.4325,  ..., -10.8709, -10.4323,  -9.6448]]],
       grad_fn=<ViewBackward0>), hidden_states=None, attentions=None)

In [10]:
from transformers import FillMaskPipeline

pipe = FillMaskPipeline(model=model, tokenizer=tokenizer)

In [11]:
pipe(text)

[{'score': 0.9478657245635986,
  'token': 1012,
  'token_str': '.',
  'sequence': 'soccer is a really fun.'},
 {'score': 0.02708783559501171,
  'token': 999,
  'token_str': '!',
  'sequence': 'soccer is a really fun!'},
 {'score': 0.024277880787849426,
  'token': 1025,
  'token_str': ';',
  'sequence': 'soccer is a really fun ;'},
 {'score': 0.0007206282461993396,
  'token': 1029,
  'token_str': '?',
  'sequence': 'soccer is a really fun?'},
 {'score': 1.5105843885976356e-05,
  'token': 1064,
  'token_str': '|',
  'sequence': 'soccer is a really fun |'}]

In [12]:
pipe('I want to [MASK] this morning')

[{'score': 0.1892286241054535,
  'token': 3342,
  'token_str': 'remember',
  'sequence': 'i want to remember this morning'},
 {'score': 0.09501578658819199,
  'token': 5293,
  'token_str': 'forget',
  'sequence': 'i want to forget this morning'},
 {'score': 0.0628630518913269,
  'token': 3637,
  'token_str': 'sleep',
  'sequence': 'i want to sleep this morning'},
 {'score': 0.05969230830669403,
  'token': 2113,
  'token_str': 'know',
  'sequence': 'i want to know this morning'},
 {'score': 0.04769374430179596,
  'token': 2707,
  'token_str': 'start',
  'sequence': 'i want to start this morning'}]

In [15]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

print(tokenizer.__class__.__name__)
print(model.__class__.__name__)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertTokenizer
BertForMaskedLM


In [16]:
from transformers import AutoTokenizer, AutoModelForNextSentencePrediction

model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForNextSentencePrediction.from_pretrained(model_name)

print(tokenizer.__class__.__name__)
print(model.__class__.__name__)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForNextSentencePrediction LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertTokenizer
BertForNextSentencePrediction


In [17]:
sentence1 = "In Italy, pizza served in formal settings, such as at a restaurant, is presented unsliced."  # 문장 A
sentence2 = "pizza is eaten with the use of a knife and fork. In casual settings, however, it is cut into wedges to be eaten while held in the hand."  # 문장 B

inputs = tokenizer(sentence1, sentence2, return_tensors='pt')
inputs

{'input_ids': tensor([[  101,  1999,  3304,  1010, 10733,  2366,  1999,  5337, 10906,  1010,
          2107,  2004,  2012,  1037,  4825,  1010,  2003,  3591,  4895, 14540,
          6610,  2094,  1012,   102, 10733,  2003,  8828,  2007,  1996,  2224,
          1997,  1037,  5442,  1998,  9292,  1012,  1999, 10017, 10906,  1010,
          2174,  1010,  2009,  2003,  3013,  2046, 17632,  2015,  2000,  2022,
          8828,  2096,  2218,  1999,  1996,  2192,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [20]:
import pandas as pd

token_sent1 = tokenizer.tokenize(sentence1)
token_sent2 = tokenizer.tokenize(sentence2)

tokens = ['[CLS]'] + token_sent1 + ['[SEP]'] + token_sent2 + ['[SEP]']
print(tokens)

pd.set_option('display.max_columns', None)

pd.DataFrame([
    tokens,
    inputs['input_ids'].squeeze(0).numpy(),
    inputs['token_type_ids'].squeeze(0).numpy(),
], index = ['tokens', 'input_ids', 'token_type_ids'])

['[CLS]', 'in', 'italy', ',', 'pizza', 'served', 'in', 'formal', 'settings', ',', 'such', 'as', 'at', 'a', 'restaurant', ',', 'is', 'presented', 'un', '##sl', '##ice', '##d', '.', '[SEP]', 'pizza', 'is', 'eaten', 'with', 'the', 'use', 'of', 'a', 'knife', 'and', 'fork', '.', 'in', 'casual', 'settings', ',', 'however', ',', 'it', 'is', 'cut', 'into', 'wedge', '##s', 'to', 'be', 'eaten', 'while', 'held', 'in', 'the', 'hand', '.', '[SEP]']


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57
tokens,[CLS],in,italy,",",pizza,served,in,formal,settings,",",such,as,at,a,restaurant,",",is,presented,un,##sl,##ice,##d,.,[SEP],pizza,is,eaten,with,the,use,of,a,knife,and,fork,.,in,casual,settings,",",however,",",it,is,cut,into,wedge,##s,to,be,eaten,while,held,in,the,hand,.,[SEP]
input_ids,101,1999,3304,1010,10733,2366,1999,5337,10906,1010,2107,2004,2012,1037,4825,1010,2003,3591,4895,14540,6610,2094,1012,102,10733,2003,8828,2007,1996,2224,1997,1037,5442,1998,9292,1012,1999,10017,10906,1010,2174,1010,2009,2003,3013,2046,17632,2015,2000,2022,8828,2096,2218,1999,1996,2192,1012,102
token_type_ids,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [23]:
output = model(**inputs)
print(output)

prob = F.softmax(output[0])
print(prob)

pred = torch.argmax(prob, dim=-1).item()
print(pred)

NextSentencePredictorOutput(loss=None, logits=tensor([[ 6.3958, -6.3766]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
tensor([[1.0000e+00, 2.8382e-06]], grad_fn=<SoftmaxBackward0>)
0


C:\Users\playdata2\AppData\Local\Temp\ipykernel_24284\4056460273.py:4: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  prob = F.softmax(output[0])


In [25]:
sentence3 = 'The Sky is blue due to the shorter wavelength of blue light'

inputs = tokenizer(sentence1, sentence3, return_tensors='pt')
output = model(**inputs)
prob = F.softmax(output[0])
pred = torch.argmax(prob, dim=-1).item()
print(prob)
print(pred)

tensor([[7.6327e-05, 9.9992e-01]], grad_fn=<SoftmaxBackward0>)
1


C:\Users\playdata2\AppData\Local\Temp\ipykernel_24284\2258279204.py:5: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  prob = F.softmax(output[0])


In [29]:
candidates = [
    sentence2,
    sentence3,
    'Pizza is one of the most popular foods in the world.',
    'i enjoy playing football on weekends.'
]

scores = []
for s2 in candidates:
    inp = tokenizer(sentence1, s2, return_tensors='pt')
    output = model(**inp)
    prob = F.softmax(output.logits, dim=-1).squeeze(0)
    print(prob)
    isnext = float(prob[0])
    scores.append((isnext, s2))

scores.sort(reverse=True, key=lambda x: x[0])

for p, s2 in scores:
    print(f'확률: {p:.4f}, 문장: {s2}')

tensor([1.0000e+00, 2.8382e-06], grad_fn=<SqueezeBackward1>)
tensor([7.6327e-05, 9.9992e-01], grad_fn=<SqueezeBackward1>)
tensor([1.0000e+00, 3.5912e-06], grad_fn=<SqueezeBackward1>)
tensor([1.8709e-05, 9.9998e-01], grad_fn=<SqueezeBackward1>)
확률: 1.0000, 문장: pizza is eaten with the use of a knife and fork. In casual settings, however, it is cut into wedges to be eaten while held in the hand.
확률: 1.0000, 문장: Pizza is one of the most popular foods in the world.
확률: 0.0001, 문장: The Sky is blue due to the shorter wavelength of blue light
확률: 0.0000, 문장: i enjoy playing football on weekends.
